<a href="https://colab.research.google.com/github/Ivan35-arch/movie_reccomender-system/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
import requests
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [4]:

url = 'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
response = requests.get(url, stream=True)

# Write the ZIP file to disk
with open('ml-latest-small.zip', 'wb') as file:
    for chunk in response.iter_content(chunk_size=1024):
        if chunk:
            file.write(chunk)

# Extract the desired CSV files
with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
    zip_ref.extractall()

# Load the extracted CSV files into DataFrames
ratings = pd.read_csv('ml-latest-small/ratings.csv', usecols=['userId', 'movieId', 'rating'])
movies = pd.read_csv('ml-latest-small/movies.csv', usecols=['movieId', 'title'])

print("Dataframes loaded successfully!")

Dataframes loaded successfully!


In [ ]:
movies

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017)
9738,193583,No Game No Life: Zero (2017)
9739,193585,Flint (2017)
9740,193587,Bungo Stray Dogs: Dead Apple (2018)


In [ ]:
ratings

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [7]:
#merge the ratings and movies dataset
data = pd.merge(ratings,movies,on= 'movieId')

#create a user item-matrix
user_item_matrix = data.pivot_table(index='userId',columns= 'title',values='rating')
user_item_matrix.fillna(0,inplace=True)

In [ ]:
data

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
100831,610,166534,4.0,Split (2017)
100832,610,168248,5.0,John Wick: Chapter Two (2017)
100833,610,168250,5.0,Get Out (2017)
100834,610,168252,5.0,Logan (2017)


In [8]:
user_item_matrix

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
607,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.5,3.5,0.0,0.0,0.0


In [9]:
#calculate cosine similarity btwn users
user_similarity= cosine_similarity(user_item_matrix)
print(user_similarity)


[[1.         0.02728287 0.05972026 ... 0.29109737 0.09357193 0.14532081]
 [0.02728287 1.         0.         ... 0.04621095 0.0275654  0.10242675]
 [0.05972026 0.         1.         ... 0.02112846 0.         0.03211875]
 ...
 [0.29109737 0.04621095 0.02112846 ... 1.         0.12199271 0.32205486]
 [0.09357193 0.0275654  0.         ... 0.12199271 1.         0.05322546]
 [0.14532081 0.10242675 0.03211875 ... 0.32205486 0.05322546 1.        ]]


In [11]:
#The code at the bottom calls the function specifically for User 4 using a default top_n of 5. It calculates who matches User 4's profile best, ignores User 4's self-match, grabs the next 5 closest indices, and prints out their actual User IDs.
#function to find similar users
def find_similar_users(user_id,user_similarity, top_n=5):
  user_index = user_item_matrix.index.get_loc(user_id)
  similar_users = user_similarity[user_index]
  similar_users_indices = np.argsort(similar_users)[::-1][1:top_n+1]
  return user_item_matrix.index[similar_users_indices]


#find the most similar users for user 4
similar_users = find_similar_users(4,user_similarity)
print(f'similar users for user 4:{similar_users}')



similar users for user 4:Index([391, 603, 156, 275, 597], dtype='int64', name='userId')


In [12]:
def generate_recommendations(user_id, user_similarity, user_item_matrix,top_n=5):
    similar_users = find_similar_users(user_id,user_similarity)
    similar_users_ratings = user_item_matrix.loc[similar_users]
    average_ratings = similar_users_ratings.mean()
    recommended_movies = average_ratings.sort_values(ascending=False).head(top_n)
    return recommended_movies

  #generate movie recommendations for user 4
recommendations = generate_recommendations(50,user_similarity, user_item_matrix)
print(f'Recommended movies for user 4: \n{recommendations}')

Recommended movies for user 4: 
title
Fight Club (1999)                   4.7
Pulp Fiction (1994)                 4.6
Godfather, The (1972)               4.5
Shawshank Redemption, The (1994)    4.4
Fargo (1996)                        4.4
dtype: float64


In [13]:
#evaluating the model
def evaluate_model(user_id, user_similarity,user_item_matrix, actual_ratings, top_n=5):
    recommendations = generate_recommendations(user_id, user_similarity, user_item_matrix, top_n)
    common_movies = recommendations.index.intersection(actual_ratings.index)
    precision = len(common_movies) / top_n
    recall = len(common_movies) / len(actual_ratings[actual_ratings > 0])#filter rated movies (above 0)
    f1_score = 2 * (precision * recall)/ (precision + recall)
    return precision, recall, f1_score

  #evaluate the model for user 4
actual_ratings = user_item_matrix.loc[4]
precision,recall,f1_score = evaluate_model(4, user_similarity, user_item_matrix, actual_ratings, top_n=5)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)


Precision: 1.0
Recall: 0.023148148148148147
F1 Score: 0.04524886877828054


In [14]:
import joblib

model_data = {
    'user_similarity':user_similarity,
    'user_item_matrix' :user_item_matrix
}

#save with compression
joblib.dump(model_data,'movie_recommender_model.pkl',compress=3)
print("Model saved.")

Model saved.


In [15]:
import joblib

loaded_model = joblib.load('movie_recommender_model.joblib')

user_similarity = loaded_model['user_similarity']
user_item_matrix = loaded_model['user_item_matrix']